# IMPORT

In [56]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

# LOAD DATA

In [57]:
rating = pd.read_csv("./data/Ratings.csv")
users = pd.read_csv("./data/Users.csv")

In [58]:
rating.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [59]:
rating.info()

<class 'pandas.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   User-ID      1149780 non-null  int64
 1   ISBN         1149780 non-null  str  
 2   Book-Rating  1149780 non-null  int64
dtypes: int64(2), str(1)
memory usage: 26.3 MB


### Create the sparse matrix

In [60]:
user_indices, user_ids =  pd.factorize(rating['User-ID'])
book_indices, book_ids = pd.factorize(rating['ISBN'])

In [61]:
user_book = csr_matrix(
    (
        rating['Book-Rating'],
        (user_indices, book_indices)
    )
)

In [62]:
user_book.shape

(105283, 340556)

### Create the target column

In [63]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [64]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 278858 entries, 0 to 278857
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   User-ID   278858 non-null  int64  
 1   Location  278858 non-null  str    
 2   Age       168096 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 6.4 MB


In [65]:
users_age = users.set_index("User-ID")['Age']
y = users_age.reindex(user_ids).reset_index(drop=True)
y

0          NaN
1          NaN
2         16.0
3         16.0
4         37.0
          ... 
105278     NaN
105279    18.0
105280    38.0
105281    14.0
105282    12.0
Name: Age, Length: 105283, dtype: float64

### Split the dataset into training, valid, and test parts

In [66]:
mask = y.notna()
X = user_book[mask.to_numpy()]
y = y[mask]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=2/3,
    random_state=21
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=21
)

### Look at the distribution of data by age